# Medical Classification: Breast Tumour Diagnosis Support

**Family:** End-to-end project | **Dataset:** `breast_cancer` | **Task:** classification

Choose a model by sensitivity at a fixed specificity target, with intervals and an error review.

## Table of Contents

- 1. Problem statement
- 2. Business or research context
- 3. Dataset documentation
- 4. Research references
- 5. Exploratory analysis
- 6. Preprocessing
- 7. Model selection
- 8. Model training
- 9. Evaluation
- 10. Comparison
- 11. Interpretation
- 12. Limitations
- 13. Final recommendations

## 1. Problem statement

Classify tumours as malignant or benign from nucleus measurements and report how the model errs.

## 2. Business or research context

In screening, a missed malignancy is worse than a false alarm. We therefore fix a sensitivity target of 0.95 on validation data and report specificity at that point. This is a **teaching example, not a medical device**.

## 3. Dataset documentation

**Breast Cancer Wisconsin (Diagnostic)**, UCI, licence CC BY 4.0, DOI 10.24432/C5DW2B. 569 rows, 30 numeric features (mean, standard error and worst value of ten nucleus properties computed from digitised fine needle aspirate images). Target `malignant` (1 malignant, 0 benign). No missing values. Limits: one institution, small sample.

## 4. Research references

Verification status is from the build environment on 2026-11-17. Entries marked *not verified* must be checked before citing.

- Wolberg, W. H. and Mangasarian, O. L. (1990). *Multisurface method of pattern separation for medical diagnosis applied to breast cytology*. PNAS 87(23):9193-9196. https://doi.org/10.1073/pnas.87.23.9193 Verification: Crossref 2026-11-17
- Cortes, C. and Vapnik, V. (1995). *Support-vector networks*. Machine Learning 20(3):273-297. https://doi.org/10.1007/BF00994018 Verification: Crossref 2026-11-17
- Breiman, L. (2001). *Random Forests*. Machine Learning 45(1):5-32. https://doi.org/10.1023/A:1010933404324 Verification: Crossref 2026-11-17

Wolberg and Mangasarian describe cytology based diagnosis from the same research group. The other two define the models compared.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score
from ml_lab.data import load_dataset
from ml_lab.preprocessing import split_three_way, make_model_pipeline, build_preprocessor
from ml_lab.evaluation import evaluate_estimator, internal_scores, external_scores
from ml_lab.experiments import ExperimentRecord, save_figure
from ml_lab.utils import set_seed, print_environment
from ml_lab import visualization as viz
PROJECT_ID = "medical_classification"
SEED = 42
set_seed(SEED)
viz.apply_style()
print_environment()

python 3.13.18 | Linux-6.18.44-fc-v595-x86_64-with-glibc2.39
  numpy==2.5.3
  pandas==3.0.5
  scipy==1.18.1
  scikit-learn==1.9.1
  matplotlib==3.11.2
  xgboost==3.4.1
  lightgbm==4.7.0
  catboost==1.2.10
  umap-learn==0.6.0
  torch==2.15.0
  gymnasium==1.4.0


In [2]:
bundle = load_dataset("breast_cancer")
X, y = bundle.X, bundle.y
Xtr, Xva, Xte, ytr, yva, yte = split_three_way(X, y, task="classification", seed=SEED)
print(len(Xtr), len(Xva), len(Xte), "| malignant share:", round(float(y.mean()), 3))

341 114 114 | malignant share: 0.373


## 5. Exploratory analysis

Class balance and the strongest single features.

In [3]:
from sklearn.metrics import roc_auc_score
single = pd.Series({c: roc_auc_score(ytr, Xtr[c]) for c in X.columns})
single = (single - 0.5).abs().sort_values(ascending=False) + 0.5
display(single.head(6).round(3).to_frame("single feature ROC-AUC"))

,single feature ROC-AUC
worst perimeter,0.972
mean concave points,0.967
worst radius,0.967
worst area,0.965
worst concave points,0.965
mean perimeter,0.947


## 6. Preprocessing

Scaling inside the pipeline. No imputation needed (no missing values).

In [4]:
print("missing values:", int(X.isna().sum().sum()))

missing values: 0


## 7. Model selection

Logistic regression, RBF support vector machine and random forest. The model is chosen by validation ROC-AUC. The threshold is then set so that validation sensitivity is at least 0.95.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
cands = {"logistic": LogisticRegression(max_iter=3000, C=1.0), "svm_rbf": SVC(probability=True, C=3, random_state=SEED), "random_forest": RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=2)}
fitted = {n: make_model_pipeline(Xtr, m).fit(Xtr, ytr) for n, m in cands.items()}
sel = pd.Series({n: roc_auc_score(yva, f.predict_proba(Xva)[:, 1]) for n, f in fitted.items()}, name="validation ROC-AUC")
display(sel.round(4).to_frame()); BEST = sel.idxmax(); print("selected:", BEST)

,validation ROC-AUC
logistic,0.9957
svm_rbf,0.9961
random_forest,0.9918


selected: svm_rbf


## 8. Model training

Refit the selected model on training plus validation rows. The threshold comes from the validation scores of the model trained on the training rows.

In [6]:
pv = fitted[BEST].predict_proba(Xva)[:, 1]
cands_t = np.unique(pv)
ok = [t for t in cands_t if ((pv >= t) & (yva == 1)).sum() / (yva == 1).sum() >= 0.95]
THRESH = float(max(ok)) if ok else 0.5
Xtv, ytv = pd.concat([Xtr, Xva]), pd.concat([ytr, yva])
final = make_model_pipeline(Xtv, clone(cands[BEST])).fit(Xtv, ytv)
print("threshold giving at least 0.95 validation sensitivity:", round(THRESH, 3))

threshold giving at least 0.95 validation sensitivity: 0.854


## 9. Evaluation

Test results with 95 percent bootstrap intervals (1000 resamples). With 114 test rows the intervals are wide, and that width is the honest uncertainty.

In [7]:
from sklearn.metrics import confusion_matrix
pt = final.predict_proba(Xte)[:, 1]
pred = (pt >= THRESH).astype(int)
tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
rng = np.random.default_rng(SEED); sens, spec, auc = [], [], []
yv = yte.values
for _ in range(1000):
    i = rng.integers(0, len(yv), len(yv))
    if yv[i].min() == yv[i].max():
        continue
    s_ = ((pred[i] == 1) & (yv[i] == 1)).sum() / (yv[i] == 1).sum(); p_ = ((pred[i] == 0) & (yv[i] == 0)).sum() / (yv[i] == 0).sum()
    sens.append(s_); spec.append(p_); auc.append(roc_auc_score(yv[i], pt[i]))
ci = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))  # noqa: E731
METRICS = {"test_ROC_AUC": roc_auc_score(yte, pt), "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp), "false_negatives": int(fn), "false_positives": int(fp),
           "sensitivity_ci_low": ci(sens)[0], "sensitivity_ci_high": ci(sens)[1], "specificity_ci_low": ci(spec)[0], "specificity_ci_high": ci(spec)[1]}
CONFIG = {"selected_model": BEST, "threshold": THRESH, "target_validation_sensitivity": 0.95}
EXTRA = {"validation_ROC_AUC": sel.round(4).to_dict(), "test_rows": int(len(yte))}
ALGORITHM = BEST
display(pd.DataFrame([[tn, fp], [fn, tp]], index=["actual benign", "actual malignant"], columns=["predicted benign", "predicted malignant"]))

record = ExperimentRecord(notebook=PROJECT_ID, family="project", algorithm=ALGORITHM, dataset="breast_cancer", task="classification", seed=SEED, config=CONFIG, metrics=METRICS, extra=EXTRA)
print("saved:", record.save())
display(pd.Series(METRICS, name="metric").round(4).to_frame())

,predicted benign,predicted malignant
actual benign,72,0
actual malignant,6,36


saved: experiments/results/medical_classification.json


,metric
test_ROC_AUC,0.9954
sensitivity,0.8571
specificity,1.0000
false_negatives,6.0000
false_positives,0.0000
sensitivity_ci_low,0.7347
sensitivity_ci_high,0.9583
specificity_ci_low,1.0000
specificity_ci_high,1.0000


## 10. Comparison

All three models at their own validation derived thresholds on the same test rows.

In [8]:
rows = []
for n, f in fitted.items():
    p = f.predict_proba(Xva)[:, 1]
    okt = [t for t in np.unique(p) if ((p >= t) & (yva == 1)).sum() / (yva == 1).sum() >= 0.95]
    t = max(okt) if okt else 0.5
    q = f.predict_proba(Xte)[:, 1] >= t
    rows.append({"model": n, "threshold": t, "test_sensitivity": ((q == 1) & (yte == 1)).sum() / (yte == 1).sum(), "test_specificity": ((q == 0) & (yte == 0)).sum() / (yte == 0).sum()})
display(pd.DataFrame(rows).set_index("model").round(3))

,threshold,test_sensitivity,test_specificity
model,,,
logistic,0.929,0.786,1.0
svm_rbf,0.854,0.905,1.0
random_forest,0.610,0.905,1.0


## 11. Interpretation

Which features drive the model, and what do the missed cases look like.

In [9]:
imp = permutation_importance(final, Xte, yte, scoring="roc_auc", n_repeats=10, random_state=SEED)
display(pd.Series(imp.importances_mean, index=Xte.columns).sort_values(ascending=False).head(6).round(4).to_frame("ROC-AUC drop"))
missed = Xte[(pred == 0) & (yte.values == 1)]
print("missed malignant rows:", len(missed)); display(missed[["radius_mean", "area_mean", "concavity_mean"]].round(3) if "radius_mean" in missed else missed.iloc[:, :3])

,ROC-AUC drop
worst texture,0.0078
worst radius,0.0051
worst area,0.0039
worst perimeter,0.0038
worst concavity,0.0038
mean concave points,0.0038


missed malignant rows: 6


,mean radius,mean texture,mean perimeter
99,14.42,19.77,94.48
73,13.80,15.79,90.43
86,14.48,21.46,94.25
385,14.60,23.29,93.97
190,14.22,23.12,94.37
205,15.12,16.68,98.78


## 12. Limitations

The validation sensitivity target of 0.95 is a design goal, not a guarantee: in the committed run the test sensitivity is lower than the target and its bootstrap interval is wide (see the results table), because a threshold chosen on about 114 validation rows does not transfer exactly. One institution, 569 rows, only 114 test rows. No external validation. The data are measurements from images, not images. The result must not be used for patient care.

## 13. Final recommendations

Treat this as a method demonstration. Real use needs external validation on another site, clinical review of error costs, calibration checks and regulatory review.